In [ ]:
# =========================
# INSTALL (run once)
# =========================
# pip install selenium pandas

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import os


# =========================
# STOCKS (Ticker + Name)
# =========================
STOCKS = {
    "SUNPHARMA": "SUN PHARMA",
    "PFC": "POWER FINANCE",
    "DRREDDY": "DR REDDY",
    "CIPLA": "CIPLA",
    "TORNTPHARM": "TORRENT PHARMA"
}


def get_screener_annual_profitloss_selenium(ticker_code):

    url = f"https://www.screener.in/company/{ticker_code}/consolidated/#profit-loss"

    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    wait = WebDriverWait(driver, 10)

    try:
        section = wait.until(
            EC.presence_of_element_located((By.ID, "profit-loss"))
        )
    except:
        driver.quit()
        raise ValueError(f"Profit-loss section not found for {ticker_code}")

    try:
        table = section.find_element(By.TAG_NAME, "table")
    except:
        driver.quit()
        raise ValueError(f"Profit-loss table not found for {ticker_code}")

    rows = table.find_elements(By.TAG_NAME, "tr")
    headers = [th.text.strip() for th in rows[0].find_elements(By.TAG_NAME, "th")]

    data = []
    for row in rows[1:]:
        cols = [td.text.strip() for td in row.find_elements(By.TAG_NAME, "td")]
        if cols:
            data.append(cols)

    driver.quit()

    df = pd.DataFrame(data, columns=headers)

    # TRANSPOSE
    df = df.set_index(headers[0]).T
    df.columns = df.columns.str.strip()

    # REMOVE GARBAGE COLUMNS
    df = df.loc[:, ~df.columns.str.match(r'^\d+\.?$')]
    df = df.dropna(axis=1, how='all')

    # CLEAN NUMERIC VALUES
    df = df.replace(",", "", regex=True)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # DATE INDEX
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.sort_index()

    return df


def standardize_columns(df):

    # Clean column names
    df.columns = (
        df.columns.str.strip()
        .str.upper()
        .str.replace("+", "", regex=False)
        .str.replace("%", "PERCENT", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    # =========================
    # MERGE SALES_ and REVENUE_
    # =========================
    if "SALES_" in df.columns and "REVENUE_" in df.columns:
        df["SALES_"] = df["SALES_"].fillna(df["REVENUE_"])
        df = df.drop(columns=["REVENUE_"])

    elif "REVENUE_" in df.columns and "SALES_" not in df.columns:
        df = df.rename(columns={"REVENUE_": "SALES_"})

    # Rename SALES_ to SALES
    if "SALES_" in df.columns:
        df = df.rename(columns={"SALES_": "SALES"})

    # =========================
    # MERGE FINANCING PROFIT with OPERATING PROFIT
    # =========================
    if "FINANCING_PROFIT" in df.columns and "OPERATING_PROFIT" in df.columns:
        df["OPERATING_PROFIT"] = df["OPERATING_PROFIT"].fillna(df["FINANCING_PROFIT"])
        df = df.drop(columns=["FINANCING_PROFIT"])

    elif "FINANCING_PROFIT" in df.columns and "OPERATING_PROFIT" not in df.columns:
        df = df.rename(columns={"FINANCING_PROFIT": "OPERATING_PROFIT"})

    # =========================
    # RENAME OTHER COMMON COLUMNS
    # =========================
    rename_map = {
        "NET_PROFIT_": "NET_PROFIT",
        "EXPENSES_": "EXPENSES",
        "OTHER_INCOME_": "OTHER_INCOME",
        "EPS_IN_RS": "EPS",
        "PROFIT_BEFORE_TAX": "PBT",
        "DIVIDEND_PAYOUT_PERCENT": "DIVIDEND_PAYOUT_PERCENT",
        "OPM_PERCENT": "OPM_PERCENT",
        "TAX_PERCENT": "TAX_PERCENT"
    }

    df = df.rename(columns=rename_map)

    return df


# =========================
# STEP 1: CREATE FILE FOR EACH TICKER
# =========================
for ticker_code, company_name in STOCKS.items():

    print(f"\nFetching data for {ticker_code} - {company_name}")

    try:
        df_annual = get_screener_annual_profitloss_selenium(ticker_code)

        # Standardize column names + merge SALES/REVENUE
        df_annual = standardize_columns(df_annual)

        # Add ticker and company name columns
        df_annual["TICKER"] = ticker_code
        df_annual["COMPANY"] = company_name

        # Move TICKER + COMPANY to first columns
        cols = ["TICKER", "COMPANY"] + [col for col in df_annual.columns if col not in ["TICKER", "COMPANY"]]
        df_annual = df_annual[cols]

        # Save file
        file_name = f"{ticker_code.lower()}_annual_profitloss.csv"
        df_annual.to_csv(file_name, index=True)

        print(f"📁 File saved: {file_name}")

    except Exception as e:
        print(f"❌ Error for {ticker_code}: {e}")


# =========================
# STEP 2: MERGE ALL FILES INTO ONE CONSOLIDATED FILE
# =========================
all_dfs = []

for ticker_code in STOCKS.keys():
    file_name = f"{ticker_code.lower()}_annual_profitloss.csv"

    if os.path.exists(file_name):
        df = pd.read_csv(file_name)
        all_dfs.append(df)
    else:
        print(f"File not found: {file_name}")

final_df = pd.concat(all_dfs, ignore_index=True)

# Convert Year column (saved as Unnamed: 0)
final_df["Unnamed: 0"] = pd.to_datetime(final_df["Unnamed: 0"], errors="coerce")

# Sort by TICKER then YEAR
final_df = final_df.sort_values(["TICKER", "Unnamed: 0"]).reset_index(drop=True)

# Rename Unnamed: 0 to YEAR
final_df = final_df.rename(columns={"Unnamed: 0": "YEAR"})

# Save consolidated file
final_df.to_csv("ALL_STOCKS_EPS.csv", index=False)

print("\n📁 Consolidated file saved: all_stocks_annual_profitloss.csv")